# 🌱 SIH25099: Crop Stress Early Warning System (Colab Runner)
### End-to-End Pipeline: Download -> Train RGB+NIR U-Net -> Evaluate & Save Checkpoint

This notebook runs on **free Google Colab T4 GPU**. It downloads the dataset directly to Colab high-speed SSD, trains the 4-channel multi-spectral U-Net, computes full metrics (Dice, IoU, Confusion Matrix), and persists the trained model directly to your Google Drive.

## 1. Mount Google Drive & Environment Setup

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive', force_remount=True)

# Destination on your Google Drive to keep checkpoints and results safe permanently
DRIVE_PROJECT_DIR = '/content/drive/MyDrive/SIH25099_Crop_Stress'
os.makedirs(DRIVE_PROJECT_DIR, exist_ok=True)
print(f"Persistent Drive Directory: {DRIVE_PROJECT_DIR}")

## 2. Check GPU & Install AWS CLI for High-Speed Download

In [ ]:
!nvidia-smi

# Install AWS CLI cleanly in Colab
!pip install -q awscli
!aws --version

## 3. High-Speed Download to Colab Local SSD (Faster & Saves Drive Space)

In [ ]:
import os

LOCAL_DATA_DIR = "/content/dataset/raw"
EXTRACT_DIR = os.path.join(LOCAL_DATA_DIR, "extracted")
ARCHIVE_PATH = os.path.join(LOCAL_DATA_DIR, "Agriculture-Vision-2021.tar.gz")

os.makedirs(LOCAL_DATA_DIR, exist_ok=True)
os.makedirs(EXTRACT_DIR, exist_ok=True)

# 1. Download if not present
if not os.path.exists(ARCHIVE_PATH) and not any('Agriculture-Vision' in d for d in os.listdir(EXTRACT_DIR) if os.path.isdir(os.path.join(EXTRACT_DIR, d))):
    print("Downloading Agriculture-Vision dataset from AWS S3 (no-sign-request)...")
    !aws s3 cp s3://intelinair-data-releases/agriculture-vision/cvpr_challenge_2021/supervised/Agriculture-Vision-2021.tar.gz {ARCHIVE_PATH} --no-sign-request

# 2. Extract if not yet extracted
if os.path.exists(ARCHIVE_PATH):
    print("Extracting archive to Colab SSD (this takes ~2-3 mins)...")
    !tar -xzf {ARCHIVE_PATH} -C {EXTRACT_DIR}
    # Clean up archive to reclaim 15GB SSD space for Colab
    !rm -f {ARCHIVE_PATH}
    print("Extraction complete and archive cleaned up!")

# 3. Locate exact dataset root folder
DATASET_ROOT = None
for root, dirs, files in os.walk(EXTRACT_DIR):
    if 'train' in dirs and 'val' in dirs:
        DATASET_ROOT = root
        break

if DATASET_ROOT is None:
    DATASET_ROOT = os.path.join(EXTRACT_DIR, "Agriculture-Vision-2021")

print(f"Active Dataset Root: {DATASET_ROOT}")


## 4. Define Paths & Verify Dataset

In [ ]:
TRAIN_RGB_DIR = os.path.join(DATASET_ROOT, "train", "images", "rgb")
TRAIN_NIR_DIR = os.path.join(DATASET_ROOT, "train", "images", "nir")
TRAIN_LABEL_DIR = os.path.join(DATASET_ROOT, "train", "labels", "nutrient_deficiency")

VAL_RGB_DIR = os.path.join(DATASET_ROOT, "val", "images", "rgb")
VAL_NIR_DIR = os.path.join(DATASET_ROOT, "val", "images", "nir")
VAL_LABEL_DIR = os.path.join(DATASET_ROOT, "val", "labels", "nutrient_deficiency")

train_files = sorted([f for f in os.listdir(TRAIN_RGB_DIR) if f.lower().endswith(('.jpg', '.jpeg'))])
val_files = sorted([f for f in os.listdir(VAL_RGB_DIR) if f.lower().endswith(('.jpg', '.jpeg'))])

print(f"Training images: {len(train_files)}")
print(f"Validation images: {len(val_files)}")

## 5. Dataset Balancing & 4-Channel (RGB + NIR) Data Pipeline

In [ ]:
import random
import numpy as np
from PIL import Image
import tensorflow as tf

IMG_SIZE = 256
BATCH_SIZE = 8

# Find positive and negative training samples
print("Indexing labels for class balancing...")
pos_files = []
neg_files = []

for f in train_files[:2000]:  # Sample fast search
    mask_name = os.path.splitext(f)[0] + ".png"
    mask_path = os.path.join(TRAIN_LABEL_DIR, mask_name)
    if os.path.exists(mask_path):
        m = np.array(Image.open(mask_path))
        if np.count_nonzero(m) > 0:
            pos_files.append(f)
        else:
            neg_files.append(f)

random.seed(42)
k = min(len(pos_files), len(neg_files), 512)
balanced_train = random.sample(pos_files, k) + random.sample(neg_files, k)
random.shuffle(balanced_train)
print(f"Selected {len(balanced_train)} balanced training samples ({k} positive, {k} negative).")

def load_rgb_nir_mask(filename, rgb_dir, nir_dir, label_dir):
    filename = tf.convert_to_tensor(filename, dtype=tf.string)
    # RGB
    rgb_p = tf.strings.join([rgb_dir, "/", filename])
    rgb = tf.image.decode_jpeg(tf.io.read_file(rgb_p), channels=3)
    rgb = tf.cast(tf.image.resize(rgb, [IMG_SIZE, IMG_SIZE]), tf.float32) / 255.0
    # NIR
    nir_p = tf.strings.join([nir_dir, "/", filename])
    nir = tf.image.decode_jpeg(tf.io.read_file(nir_p), channels=1)
    nir = tf.cast(tf.image.resize(nir, [IMG_SIZE, IMG_SIZE]), tf.float32) / 255.0
    # 4 channels
    x = tf.concat([rgb, nir], axis=-1)
    # Label mask
    mask_name = tf.strings.regex_replace(filename, r"\.(jpg|jpeg)$", ".png")
    mask_p = tf.strings.join([label_dir, "/", mask_name])
    mask = tf.image.decode_png(tf.io.read_file(mask_p), channels=1)
    mask = tf.cast(tf.image.resize(mask, [IMG_SIZE, IMG_SIZE], method="nearest") > 127, tf.float32)
    return x, mask

def make_ds(files, rgb_dir, nir_dir, label_dir, training=False):
    ds = tf.data.Dataset.from_tensor_slices(files)
    if training:
        ds = ds.shuffle(1000, reshuffle_each_iteration=True)
    ds = ds.map(lambda f: load_rgb_nir_mask(f, rgb_dir, nir_dir, label_dir), num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds = make_ds(balanced_train, TRAIN_RGB_DIR, TRAIN_NIR_DIR, TRAIN_LABEL_DIR, training=True)
val_ds = make_ds(val_files[:400], VAL_RGB_DIR, VAL_NIR_DIR, VAL_LABEL_DIR, training=False)
print("tf.data pipelines ready!")

## 6. Build 4-Channel U-Net & Combined Loss (Dice + BCE)

In [ ]:
from tensorflow.keras import layers, models

def build_unet(input_shape=(256, 256, 4)):
    inputs = layers.Input(input_shape)
    c1 = layers.Conv2D(32, 3, activation="relu", padding="same")(inputs)
    c1 = layers.Conv2D(32, 3, activation="relu", padding="same")(c1)
    p1 = layers.MaxPooling2D((2, 2))(c1)

    c2 = layers.Conv2D(64, 3, activation="relu", padding="same")(p1)
    c2 = layers.Conv2D(64, 3, activation="relu", padding="same")(c2)
    p2 = layers.MaxPooling2D((2, 2))(c2)

    c3 = layers.Conv2D(128, 3, activation="relu", padding="same")(p2)
    c3 = layers.Conv2D(128, 3, activation="relu", padding="same")(c3)
    p3 = layers.MaxPooling2D((2, 2))(c3)

    b = layers.Conv2D(256, 3, activation="relu", padding="same")(p3)
    b = layers.Conv2D(256, 3, activation="relu", padding="same")(b)

    u3 = layers.UpSampling2D((2, 2))(b)
    u3 = layers.Concatenate()([u3, c3])
    c4 = layers.Conv2D(128, 3, activation="relu", padding="same")(u3)
    c4 = layers.Conv2D(128, 3, activation="relu", padding="same")(c4)

    u2 = layers.UpSampling2D((2, 2))(c4)
    u2 = layers.Concatenate()([u2, c2])
    c5 = layers.Conv2D(64, 3, activation="relu", padding="same")(u2)
    c5 = layers.Conv2D(64, 3, activation="relu", padding="same")(c5)

    u1 = layers.UpSampling2D((2, 2))(c5)
    u1 = layers.Concatenate()([u1, c1])
    c6 = layers.Conv2D(32, 3, activation="relu", padding="same")(u1)
    c6 = layers.Conv2D(32, 3, activation="relu", padding="same")(c6)

    outputs = layers.Conv2D(1, 1, activation="sigmoid")(c6)
    return models.Model(inputs, outputs)

def dice_loss(y_true, y_pred, smooth=1e-6):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    axes = [1, 2, 3]
    intersection = tf.reduce_sum(y_true * y_pred, axis=axes)
    denom = tf.reduce_sum(y_true, axis=axes) + tf.reduce_sum(y_pred, axis=axes)
    dice = (2.0 * intersection + smooth) / (denom + smooth)
    return 1.0 - tf.reduce_mean(dice)

def combined_loss(y_true, y_pred):
    return tf.reduce_mean(tf.keras.losses.binary_crossentropy(y_true, y_pred)) + dice_loss(y_true, y_pred)

model = build_unet(input_shape=(256, 256, 4))
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss=combined_loss,
    metrics=[tf.keras.metrics.BinaryAccuracy(name="binary_accuracy")]
)
print("Model compiled!")

## 7. Train with Auto-Save Checkpoints Directly into Google Drive

In [ ]:
DRIVE_BEST_MODEL = os.path.join(DRIVE_PROJECT_DIR, "unet_rgb_nir_best.keras")
DRIVE_LATEST_MODEL = os.path.join(DRIVE_PROJECT_DIR, "unet_rgb_nir_latest.keras")

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(DRIVE_BEST_MODEL, monitor="val_loss", save_best_only=True, verbose=1),
    tf.keras.callbacks.ModelCheckpoint(DRIVE_LATEST_MODEL, save_best_only=False, verbose=0),
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, verbose=1)
]

EPOCHS = 20
print(f"Starting training for {EPOCHS} epochs...")
history = model.fit(
    train_ds,
    epochs=EPOCHS,
    validation_data=val_ds,
    callbacks=callbacks,
    verbose=1
)
print(f"Training complete! Model saved to Google Drive: {DRIVE_BEST_MODEL}")

## 8. Final Evaluation: Dice, IoU, Precision, Recall & Confusion Matrix

In [ ]:
import matplotlib.pyplot as plt

all_preds, all_masks = [], []
for b_img, b_msk in val_ds.take(15):
    all_preds.append(model.predict(b_img, verbose=0))
    all_masks.append(b_msk.numpy())

yp_all = np.concatenate(all_preds, axis=0)
yt_all = np.concatenate(all_masks, axis=0)

print('=' * 65)
print('RGB + NIR EVALUATION METRICS')
print('=' * 65)
print(f'{"Threshold":<10} | {"Dice (F1)":<10} | {"IoU":<10} | {"Precision":<10} | {"Recall":<10}')
print('-' * 65)

for th in [0.2, 0.3, 0.4, 0.5]:
    yt = (yt_all > 0.5).astype(np.uint8).flatten()
    yp = (yp_all >= th).astype(np.uint8).flatten()
    tp = np.sum((yt == 1) & (yp == 1))
    fp = np.sum((yt == 0) & (yp == 1))
    fn = np.sum((yt == 1) & (yp == 0))
    tn = np.sum((yt == 0) & (yp == 0))
    eps = 1e-6
    prec = (tp + eps) / (tp + fp + eps)
    rec = (tp + eps) / (tp + fn + eps)
    dice = (2 * tp + eps) / (2 * tp + fp + fn + eps)
    iou = (tp + eps) / (tp + fp + fn + eps)
    print(f'{th:<10.2f} | {dice:<10.4f} | {iou:<10.4f} | {prec:<10.4f} | {rec:<10.4f}')

# Confusion Matrix Plot
cm = np.array([[tn, fp], [fn, tp]])
plt.figure(figsize=(5, 4))
plt.imshow(cm, cmap="Blues")
plt.title("Pixel Confusion Matrix (Th=0.50)")
plt.xticks([0, 1], ["Healthy", "Deficient"])
plt.yticks([0, 1], ["Healthy", "Deficient"])
for r in range(2):
    for c in range(2):
        plt.text(c, r, f"{cm[r, c]:,}", ha="center", va="center",
                 color="white" if cm[r, c] > cm.max()/2 else "black", fontweight="bold")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()

## 9. Prediction Visualizations (RGB, NIR, Ground Truth, Predicted Mask, Overlay)

In [ ]:
fig, axes = plt.subplots(4, 5, figsize=(18, 16))
count = 0

for images, masks in val_ds.take(1):
    preds = model.predict(images, verbose=0)
    for i in range(min(4, len(images))):
        rgb = images[i].numpy()[:, :, :3]
        nir = images[i].numpy()[:, :, 3]
        gt = masks[i].numpy().squeeze()
        pred = (preds[i].squeeze() >= 0.4).astype(np.uint8)
        overlay = rgb.copy()
        overlay[pred == 1] = [1.0, 0.0, 0.0]

        axes[i, 0].imshow(rgb); axes[i, 0].set_title(f"Sample {i+1}: RGB"); axes[i, 0].axis("off")
        axes[i, 1].imshow(nir, cmap="gray"); axes[i, 1].set_title("NIR Channel"); axes[i, 1].axis("off")
        axes[i, 2].imshow(gt, cmap="gray"); axes[i, 2].set_title("Ground Truth"); axes[i, 2].axis("off")
        axes[i, 3].imshow(pred, cmap="Reds", vmin=0, vmax=1); axes[i, 3].set_title("Prediction (Th=0.4)"); axes[i, 3].axis("off")
        axes[i, 4].imshow(overlay); axes[i, 4].set_title("Stress Overlay"); axes[i, 4].axis("off")

plt.tight_layout()
plt.show()